# GEE Pipeline — Clean + Rename + Merge

Run cells **1 → 2 → 3** in order.

**Output:** `Merged_Lake_Water_Quality_Master.csv` — **353 rows, 39 lakes**

In [4]:
# ═══════════════════════════════════════════
# CELL 1 — CLEAN GEE CSV
# ═══════════════════════════════════════════
import pandas as pd, re

GEE_CSV    = '../Analysis/GEE-Dataset.csv'
MASTER_CSV = '../Analysis/master_dataset.csv'
OUT_CLEAN  = 'GEE-Dataset-Cleaned.csv'
OUT_FINAL  = 'GEE-Dataset-Cleaned_Final.csv'
OUT_MERGED = 'Merged_Lake_Water_Quality_Master.csv'

df = pd.read_csv(GEE_CSV)
print(f'Loaded              : {df.shape}')

# Auto-detect value columns (everything except ID/flag columns)
ID_COLS  = ['system:index','LAKE_NAME','MONTH_YEAR','YEAR','MONTH','n_s2_images','data_flag']
VAL_COLS = [c for c in df.columns if c not in ID_COLS]

# Drop rows where ALL value columns are NaN (no S2 data that month)
df = df.dropna(subset=VAL_COLS, how='all')
print(f'After drop all-NaN  : {df.shape}')

# Drop rows with zero valid water pixels
count_col = next((c for c in df.columns if c.endswith('_count')), None)
if count_col:
    df = df[df[count_col] > 0]
    print(f'After drop count==0 : {df.shape}  [{count_col}]')

# Drop admin-only columns
df = df.drop(columns=[c for c in ['system:index','data_flag'] if c in df.columns])

df.to_csv(OUT_CLEAN, index=False)
print(f'Saved → {OUT_CLEAN}')
df.head()


Loaded              : (4525, 31)
After drop all-NaN  : (4525, 31)
After drop count==0 : (1688, 31)  [DO_est_mgL_count]
Saved → GEE-Dataset-Cleaned.csv


,LAKE_NAME,MONTH_YEAR,YEAR,MONTH,n_s2_images,TurbidityIndex_mean,TurbidityIndex_stdDev,NDCI_mean,NDCI_stdDev,NDCI_pos_mean,...,COD_est_mgL_stdDev,TDS_est_mgL_mean,TSS_est_mgL_mean,WQI_proxy_mean,DO_clarity_composite_mean,DO_swir_ndwi_product_mean,BOD_algal_composite_mean,BOD_turbid_algal_mean,BOD_COD_sum_mean,Stress_Index_mean
182,Doddebele,08/23,2023,8,1,0.716522,0.011836,0.248601,0.008470,0.248601,...,1.125507,211.361986,49.245823,29.445684,NaN,NaN,NaN,NaN,NaN,NaN
199,Mallathahalli Lake,08/23,2023,8,1,0.906634,0.043476,0.122388,0.043832,0.122388,...,2.916188,380.019861,49.747012,26.640293,NaN,NaN,NaN,NaN,NaN,NaN
201,Muppatu Kavalu Hosa Kere,08/23,2023,8,1,0.711580,0.015080,0.263208,0.014397,0.263208,...,1.193269,208.483072,53.696266,30.131394,NaN,NaN,NaN,NaN,NaN,NaN
202,Nayandanahalli kere,08/23,2023,8,1,0.681944,0.000000,0.196399,0.000000,0.196399,...,0.000000,332.493219,54.977466,26.634312,NaN,NaN,NaN,NaN,NaN,NaN
203,Sankey Tank,08/23,2023,8,1,0.725709,0.038475,0.074732,0.049312,0.076888,...,3.329215,307.213546,45.709284,20.985965,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# ═══════════════════════════════════════════
# CELL 2 — RENAME LAKE NAMES
# ═══════════════════════════════════════════
import pandas as pd

df_gee = pd.read_csv(OUT_CLEAN)

rename_map = {
    'Mahadevepura'             : 'Mahadevpura',
    'Agara'                    : 'Agaram',
    'Allasandra'               : 'Allalasandra',
    'Bhoganahalli'             : 'Bhoganhalli',
    'Chellakere'               : 'Chelekere',
    'Devarabisanahalli'        : 'Devarabeesanahalli',
    'Gangashetti'              : 'Gangashetty',
    'Nagawara'                 : 'Nagavara',
    'Panattur'                 : 'Panathur',
    'Parapana'                 : 'Parappana Agrahara',
    'Sarraki'                  : 'Sarakki',
    'Sowl'                     : 'Soulkere',
    'Vibhutipura'              : 'Vibhuthipura',
    'Vijinapura'               : 'Vijanapura',
    'Arekere'                  : 'Arakere',
    'Thubarahalli'             : 'Tubarahalli',
    'Gangashetti / Devasandra' : 'Devasandra',
    'Kembathahali'             : 'Kembhatahali',
    'Nallurahallikere'         : 'Nallurahalli',
    'Doraikere'                : 'Uttarahalli Doraikere',
    # add more here as needed
}

df_gee['LAKE_NAME'] = (
    df_gee['LAKE_NAME']
    .map(lambda x: rename_map.get(str(x).strip(), str(x).strip()))
    .str.title().str.strip()
)

df_gee.to_csv(OUT_FINAL, index=False)
print(f'Rows: {len(df_gee)}  Unique lakes: {df_gee["LAKE_NAME"].nunique()}')
print(f'Saved → {OUT_FINAL}')


Rows: 1688  Unique lakes: 167
Saved → GEE-Dataset-Cleaned_Final.csv


In [6]:
# ═══════════════════════════════════════════
# CELL 3 — MERGE WITH MASTER DATASET
# Inner join on LAKE_NAME + MONTH_YEAR
# ═══════════════════════════════════════════
import pandas as pd, re

df_gee    = pd.read_csv(OUT_FINAL)
df_master = pd.read_csv(MASTER_CSV)

print(f'GEE rows    : {len(df_gee)}')
print(f'Master rows : {len(df_master)}')

# ── Normalise lake names for matching (applied to BOTH sides) ────────────────
def norm_name(s):
    """Lowercase, strip lake/kere/tank noise words, collapse spaces."""
    if pd.isna(s): return ''
    s = str(s).lower()
    s = re.sub(r'\(un-named lake on map\)', '', s)
    s = re.sub(r'\b(lake|kere|tank)\b', '', s)
    s = re.sub(r'[-_/]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# Rename master lake column
df_master = df_master.rename(columns={'NAMEOFMONITORINGLOCATION': 'LAKE_NAME'})

# Create normalised matching keys (do NOT overwrite display columns)
df_gee['_key_name']    = df_gee['LAKE_NAME'].map(norm_name)
df_master['_key_name'] = df_master['LAKE_NAME'].map(norm_name)
df_gee['_key_month']    = df_gee['MONTH_YEAR'].astype(str).str.strip()
df_master['_key_month'] = df_master['MONTH_YEAR'].astype(str).str.strip()

# ── MERGE — inner join (original logic, keys are _key_name + _key_month) ─────
merged_df = pd.merge(
    df_gee, df_master,
    on=['_key_name', '_key_month'],
    how='inner',
    suffixes=('_GEE', '_master')
)

# Drop helper key columns
merged_df = merged_df.drop(columns=['_key_name','_key_month'], errors='ignore')

print(f'\nMerged rows   : {len(merged_df)}')
print(f'Merged cols   : {merged_df.shape[1]}')
print(f'Matched lakes : {merged_df["LAKE_NAME_GEE"].nunique()}')

if merged_df.empty:
    print('\n⚠ No rows matched.')
    print('GEE    keys:', sorted(df_gee['_key_name'].unique())[:10])
    print('Master keys:', sorted(df_master['_key_name'].unique())[:10])
else:
    show = [c for c in ['LAKE_NAME_GEE','MONTH_YEAR_GEE','DO_est_mgL_mean','BOD_est_mgL_mean','DO','BOD'] if c in merged_df.columns]
    print(merged_df[show].head(10).to_string())

merged_df.to_csv(OUT_MERGED, index=False)
print(f'\nSaved → {OUT_MERGED}')


GEE rows    : 1688
Master rows : 2924

Merged rows   : 362
Merged cols   : 64
Matched lakes : 39
         LAKE_NAME_GEE MONTH_YEAR_GEE  DO_est_mgL_mean  BOD_est_mgL_mean   DO   BOD
0          Sankey Tank          08/23         3.338722          4.228954  4.9   7.0
1         Yediyur Kere          08/23         3.581990          4.892110  4.2  11.0
2           Begur Lake          08/23         2.772085          5.626537  5.4   6.0
3       Yelahanka Lake          08/23         3.265446          4.674028  3.0  13.0
4       Singapura Lake          08/23         2.004975          6.157776  5.4   4.0
5  Chikkabanavara Kere          08/23         3.135960          7.452552  5.3   7.3
6         Lalbagh Kere          08/23         3.263829          6.154702  4.1  12.0
7   Kasavanahalli Kere          08/23         3.565334          3.330887  4.6   8.0
8         Hebbal  Kere          08/23         3.231470          4.825918  4.2  11.0
9         Nallurahalli          08/23         2.811755         